# FakeGuard — Exploratory Data Analysis
**6INTELSY Final Project | AY 2025-2026 | Holy Angel University**

This notebook explores the LIAR dataset: label distributions, text length statistics, speaker and party breakdowns, and vocabulary analysis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')

COLUMNS = [
    'id','label','statement','subject','speaker','job_title',
    'state','party','barely_true_count','false_count',
    'half_true_count','mostly_true_count','pants_fire_count','context'
]
LABEL_MAP = {
    'pants-fire': 0, 'false': 0, 'barely-true': 0,
    'half-true':  1, 'mostly-true': 1, 'true': 1
}

def load(split):
    path = f'data/raw/{split}.tsv'
    df = pd.read_csv(path, sep='\t', header=None, names=COLUMNS)
    df['binary_label'] = df['label'].map(LABEL_MAP)
    df['text_len'] = df['statement'].str.split().str.len()
    return df.dropna(subset=['binary_label'])

train = load('train')
val   = load('valid')
test  = load('test')
print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

In [ ]:
# Label distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train['label'].value_counts().plot(kind='bar', ax=axes[0], title='6-class Labels (Train)', color='steelblue')
train['binary_label'].value_counts().rename({0:'Fake',1:'Real'}).plot(
    kind='bar', ax=axes[1], title='Binary Labels (Train)', color=['tomato','steelblue'])
plt.tight_layout()
os.makedirs('experiments/results', exist_ok=True)
plt.savefig('experiments/results/label_distribution.png', dpi=150)
plt.show()

In [ ]:
# Text length distribution
plt.figure(figsize=(10, 4))
sns.histplot(train['text_len'], bins=40, kde=True, color='steelblue')
plt.title('Statement Length Distribution (words) — Train')
plt.xlabel('Word Count')
plt.savefig('experiments/results/text_length_dist.png', dpi=150)
plt.show()
print(train['text_len'].describe())

In [ ]:
# Speaker and party breakdown
print('Top 10 speakers:')
print(train['speaker'].value_counts().head(10))
print('\nTop 10 parties:')
print(train['party'].value_counts().head(10))

In [ ]:
# Fake vs Real rate by party (slice analysis preview)
party_df = train.groupby('party')['binary_label'].mean().sort_values()
party_df.plot(kind='barh', figsize=(8,6), title='Real-news Rate by Party Affiliation', color='steelblue')
plt.xlabel('Proportion labeled Real')
plt.tight_layout()
plt.savefig('experiments/results/party_slice.png', dpi=150)
plt.show()